In [2]:
from pyspark.sql import functions as F

# ============================================================
# NB_Gold_Order_Reconciliation
# Compare source order totals against calculated sales totals
# Grain: one row per order
# ============================================================

TARGET_TABLE = "gold_order_reconciliation"

# ------------------------------------------------------------
# 1. Load Gold facts
# ------------------------------------------------------------

orders = spark.table("fact_orders")

sales = spark.table("fact_sales")

# ------------------------------------------------------------
# 2. Aggregate sales at order level
# ------------------------------------------------------------

sales_by_order = (
    sales
    .groupBy("order_id")
    .agg(
        F.round(
            F.sum("sales_amount"),
            2
        ).alias("calculated_sales_amount")
    )
)

# ------------------------------------------------------------
# 3. Join order totals with calculated sales
# ------------------------------------------------------------

reconciliation = (
    orders
    .select(
        "order_id",
        "customer_key",
        "date_key",
        "order_date",
        "status",
        "total_amount"
    )
    .join(
        sales_by_order,
        on="order_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 4. Calculate difference
# ------------------------------------------------------------

reconciliation = (
    reconciliation
    .withColumn(
        "difference",
        F.round(
            F.col("total_amount") -
            F.col("calculated_sales_amount"),
            2
        )
    )
    .withColumn(
        "reconciliation_status",
        F.when(
            F.col("calculated_sales_amount").isNull(),
            F.lit("NO_SALES_ITEMS")
        )
        .when(
            F.abs(F.col("difference")) <= F.lit(0.01),
            F.lit("MATCH")
        )
        .otherwise(
            F.lit("MISMATCH")
        )
    )
)

# ------------------------------------------------------------
# 5. Write reconciliation table
# ------------------------------------------------------------

reconciliation.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(TARGET_TABLE)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print(f"Table created: {TARGET_TABLE}")

reconciliation.groupBy(
    "reconciliation_status"
).count().orderBy(
    "reconciliation_status"
).show()

# ------------------------------------------------------------
# 7. Financial summary
# ------------------------------------------------------------

reconciliation.select(
    F.count("*").alias("total_orders"),
    F.sum(
        F.when(
            F.col("reconciliation_status") == "MATCH",
            1
        ).otherwise(0)
    ).alias("matching_orders"),
    F.sum(
        F.when(
            F.col("reconciliation_status") == "MISMATCH",
            1
        ).otherwise(0)
    ).alias("mismatching_orders"),
    F.round(
        F.sum("total_amount"),
        2
    ).alias("source_order_value"),
    F.round(
        F.sum("calculated_sales_amount"),
        2
    ).alias("calculated_sales_value"),
    F.round(
        F.sum("difference"),
        2
    ).alias("total_difference")
).show()

# ------------------------------------------------------------
# 8. Show largest discrepancies
# ------------------------------------------------------------

reconciliation.orderBy(
    F.abs(F.col("difference")).desc()
).show(10, truncate=False)

StatementMeta(, 29699d56-0e28-4ba0-80f8-582603e1d3ea, 4, Finished, Available, Finished, False)

Table created: gold_order_reconciliation
+---------------------+-----+
|reconciliation_status|count|
+---------------------+-----+
|                MATCH|    1|
|             MISMATCH|45441|
|       NO_SALES_ITEMS| 4538|
+---------------------+-----+

+------------+---------------+------------------+------------------+----------------------+----------------+
|total_orders|matching_orders|mismatching_orders|source_order_value|calculated_sales_value|total_difference|
+------------+---------------+------------------+------------------+----------------------+----------------+
|       49980|              1|             45441|      125331289.20|          358229521.67|   -244225154.12|
+------------+---------------+------------------+------------------+----------------------+----------------+

+--------+------------+--------+-------------------+---------+------------+-----------------------+----------+---------------------+
|order_id|customer_key|date_key|order_date         |status   |total_a